In [17]:
"""
Event Similarity System using Sentence Embeddings
Converts each event description to embedding vector and compares with user input
"""

import json
import numpy as np
from typing import List, Dict, Tuple, Optional
from sentence_transformers import SentenceTransformer, util
import pandas as pd


class EmbeddingSimilaritySystem:
    """
    System that converts event descriptions to embeddings and compares them
    """
    
    def __init__(self, model_name: str = '../pre_trained_models/Arabic-SBERT-100K'):
        
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.events_data = []  # Store original event data
        self.event_embeddings = None  # Store all embeddings as matrix
        print("Model loaded successfully!")
    
    def load_and_encode(self, json_file_path: str) -> None:
        
        # Load JSON data
        print(f"Loading data from {json_file_path}...")
        with open(json_file_path, 'r', encoding='utf-8') as f:
            self.events_data = json.load(f)
        
        print(f"Loaded {len(self.events_data)} events")
        
        # Extract all descriptions
        descriptions = [event['description'] for event in self.events_data]
        
        # Convert all descriptions to embeddings at once (much faster)
        print("Converting descriptions to embeddings...")
        self.event_embeddings = self.model.encode(
            descriptions, 
            convert_to_numpy=True,
            show_progress_bar=True
        )
        
        print(f"Created embeddings matrix of shape: {self.event_embeddings.shape}")
        print(f"Each description is now a {self.event_embeddings.shape[1]}-dimensional vector")

    def encode_user_input(self, user_input: str) -> np.ndarray:
        
        user_embedding = self.model.encode(user_input, convert_to_numpy=True)
        return user_embedding
    
    def compare_with_events(self, user_input: str, top_k: int = 1, threshold: float = 0.0) -> List[Dict]:
        
        if self.event_embeddings is None:
            raise ValueError("No events loaded. Call load_and_encode() first.")
        
        # Convert user input to embedding
        user_embedding = self.encode_user_input(user_input)
        
        # Calculate cosine similarity with all event embeddings
        # This gives similarity scores for all events at once
        similarities = util.cos_sim(user_embedding, self.event_embeddings)[0]
        
        # Get indices of top K similarities
        # Convert to numpy for argsort
        similarities_np = similarities.numpy() if hasattr(similarities, 'numpy') else similarities
        
        # Get top K indices
        top_indices = similarities_np.argsort()[-top_k:][::-1]
        
        # Build results
        results = []
        for idx in top_indices:
            similarity_score = float(similarities_np[idx])
            
            if similarity_score >= threshold:
                event = self.events_data[idx]
                results.append({
                    'id': event['id'],
                    'event_name': event['event_name'],
                    'description': event['description'],
                    'results': event['results'],
                    'causes': event['causes'],
                    'similarity_score': similarity_score,
                    'era': event.get('era', ''),
                    'date': event.get('date', '')
                })
        
        return results


In [18]:
# example_usage.py

import json
import pandas as pd
# from embedding_system import EmbeddingSimilaritySystem

# ============================================================
# Example 1: Load JSON and compare
# ============================================================

# Initialize the system
system = EmbeddingSimilaritySystem()

# Load your dataset and convert all descriptions to embeddings
system.load_and_encode("../datasets/events_dataset_v2.json")

# Compare user input with all events
user_input = "بناء قصر الزهراء"
results = system.compare_with_events(user_input, top_k=3)

print("="*70)
print(f"User Input: {user_input}")
print("="*70)

for i, result in enumerate(results, 1):
    print(f"\n{i}. Event: {result['event_name']}")
    print(f"   Similarity: {result['similarity_score']:.2%}")
    print(f"   Description: {result['description']}")
    print(f"   era: {result['era']}")
    print(f"   causes: {result['causes']}")
    print(f"   results: {result['results']}")


Loading embedding model: ../pre_trained_models/Arabic-SBERT-100K...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 754.93it/s, Materializing param=pooler.dense.weight]                               


Model loaded successfully!
Loading data from ../datasets/events_dataset_v2.json...
Loaded 992 events
Converting descriptions to embeddings...


Batches: 100%|██████████| 31/31 [00:00<00:00, 35.63it/s]


Created embeddings matrix of shape: (992, 768)
Each description is now a 768-dimensional vector
User Input: بناء قصر الزهراء

1. Event: بناء مدينة الزهراء (تكملة)
   Similarity: 75.42%
   Description: أمر عبد الرحمن الناصر ببناء مدينة الزهراء الفخمة، تخليداً لذكرى زوجته المفضلة، وكمقر جديد للخلافة.
   era: حقبة النشأة
   causes: ['حب الناصر لزوجته الزهراء', 'الرغبة في إظهار عظمة الدولة الأموية']
   results: ['إنشاء أعجوبة معمارية فريدة', 'نقل مقر الحكم إلى المدينة الجديدة']

2. Event: بناء قصر الزهراء
   Similarity: 69.58%
   Description: أمر عبد الرحمن الناصر ببناء قصر الزهراء ليكون مقرًا للخلافة وقصرًا للحكم شمال غرب قرطبة.
   era: حقبة الخلافة الأموية
   causes: ['الرغبة في إنشاء عاصمة جديدة تعكس عظمة الخلافة']
   results: ['إنشاء مجمع معماري فريد', 'انتقال الخلفاء إلى المدينة الجديدة']

3. Event: بناء قصر الزهراء بأمر عبد الرحمن الناصر
   Similarity: 68.74%
   Description: أمر الخليفة عبد الرحمن الناصر ببناء مدينة الزهراء (على بعد 8 كم من قرطبة) لتكون مقرًا للخلافة وقصرًا للحكم، تخ